## 1.1 后训练
**预训练在学世界，后训练在学决策。**

同样一个 prompt，Base Model 可能知道很多相关知识，但  **Post-train 决定它最终走哪条输出路径**。

第一个是我们耳熟能详的监督微调（SFT），用的是高质量 ((q, a)) 或多轮对话示例。它的核心作用不是让模型凭空长新知识，而是把 Base Model 原本散着的能力，压到一种更可用的接口上：

* 看见用户提问，就开始回答而不是散文续写
* 知道该简洁还是该展开
* 知道某些特殊格式（比如先推理、后作答）
* 知道安全边界和对话礼仪

然而，SFT 本质上还是**教师强制式模仿**。
1. **分布偏移问题**：不管模型上一步生成了什么，下一步都给它正确的token当输入；但推理时模型是自回归生成，上一步错了，下一步输入就错了，误差会持续累积。SFT从未在「错误的中间状态」上训练过，遇到长推理、新题型直接崩盘。而RL让模型自己生成完整序列，在自己采样的状态上学习，天然解决分布偏移。
2. **无法优化延迟奖励**：SFT是每一步token都做交叉熵损失，要求每一步都模仿标准答案。但数学/代码这类任务，「对错」只有生成完整序列后才知道——中间某一步推理token，单独看没有对错，只有放到完整解题路径里才有意义。SFT无法完成「信用分配」（把最终的对错奖励，分配到每一步的推理动作上），而RL的核心就是解决信用分配。
3. **只能模仿，无法探索**：SFT只能学训练集里有的示范，训练集没有的解题思路，模型永远学不会。而RL让模型自主采样不同的生成路径，通过奖励反馈探索出训练集之外的更优解法，这也是DeepSeek-R1能突破SFT上限的核心。

这就引出 PPO 和 DPO：**要把“好不好”这个反馈，显式地塞回训练里。**

> 截止我写下这里为止，强化学习的知识我只自学恶补到了TRPO以前，对于PPO还是不太懂，所以这里再额外补一下。

| 经典RL术语 | LLM场景下的精准定义 | 核心本质 |
|------------|----------------------|----------|
| 智能体(Agent) | 我们要优化的大语言模型（Policy/Actor模型） | 做决策的主体，负责生成文本 |
| 环境(Environment) | 模型的上下文窗口+Tokenizer+自回归生成规则 | 智能体所处的交互场景，决定状态的转移 |
| 状态(State, Sₜ) | t时刻的输入上下文，即前t个token组成的完整文本序列 | 比如prompt是「1+1等于几？」，生成第一个token「2」后，状态就变成了「1+1等于几？2」 |
| 动作(Action, Aₜ) | t时刻模型输出的下一个token | 模型在当前状态下，从词表中选择一个token输出，就是一次动作决策 |
| 策略(Policy, π_θ(a\s)) | 语言模型本身！θ是模型参数，π_θ(a\s)是给定上下文s，模型输出下一个token a的概率分布 | 预训练/SFT/RL，本质都是在优化这个策略：预训练拟合互联网文本分布，SFT拟合问答示范分布，RL拟合最大化奖励的分布 |
| 奖励(Reward, R) | 给模型生成的完整文本序列打的分，告诉模型「这个结果好不好」 | RLHF里是奖励模型的偏好分，数学题里是答案是否正确的客观分 |
| 累计回报(Return, G) | 一个完整生成序列的总奖励 | RL的终极目标：让模型的策略最大化期望累计回报E[G] |
| 优势函数(Advantage, Âₜ) | 「在状态sₜ下，采取动作aₜ，比平均水平好多少」 | 核心作用是降低梯度方差，告诉模型：这个动作该提权还是降权 |

### RLHF
RLHF不是凭空出现的，它是严格基于前面的SFT做的延伸:
#### 阶段1：有监督微调（SFT）
- 核心目标：给RL提供一个「靠谱的初始策略」。直接用Base模型做RL，生成的内容杂乱无章，人类无法打分，奖励模型也学不到有效偏好，RL完全无法收敛。SFT先把模型拉进「人类可接受的输出分布」，再做偏好优化。
#### 阶段2：训练奖励模型（Reward Model, RM）
- 核心目标：把人类的主观偏好，转化成可自动、批量打分的模型，替代人类给RL提供持续的奖励信号。
- 完整实现步骤：
  1. 收集大量prompt，每个prompt用SFT模型生成4-10个不同的回答；
  2. 标注员给同一个prompt的多个回答做**偏好排序**（而非绝对打分），比如A比B好，B比C好，排序比打分的标注一致性更高、更稳定；
  3. 用排序数据训练RM：RM用SFT模型初始化，把最后的语言模型头（lm_head）换成输出单个标量的线性层，输入是「prompt+完整回答」，输出是偏好分数score。
  4. 训练损失用**成对排序损失**：
     $$L_{RM} = - \mathbb{E}_{(x,y_w,y_l) \sim D} \left[ \log \sigma \left( RM(x,y_w) - RM(x,y_l) \right) \right]$$
     其中x是prompt，y_w是人类偏好的回答，y_l是不偏好的回答，σ是sigmoid函数。核心逻辑：让RM给偏好回答打的分，显著高于不偏好的回答。
#### 阶段3：用PPO算法，基于RM的奖励优化SFT模型
- 核心目标：让SFT模型的生成结果，越来越符合人类偏好（RM的高分），同时不偏离原模型太远，避免模式崩溃。

### PPO
不论如何，策略梯度我还是学到了的。Policy Gradient 的公式是：

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t \right]
$$

如果某个动作a_t之后的累计回报G_t是正的，就提高这个动作的概率；如果是负的，就降低这个动作的概率。

但它有一个无解的问题：**更新步长完全不可控**。每次参数更新后，新策略和旧策略的分布差距会极大，导致原来采样的序列数据，对新策略来说完全无效，梯度不准，训练极其不稳定，甚至直接让模型输出乱码、重复文本（模式崩溃）。

为了解决PG的步长问题，TRPO算法提出了「信赖域」：每次参数更新，必须让新策略π_new和旧策略π_old的KL散度≤δ（一个极小的阈值），保证策略更新不会太猛。但TRPO需要二阶优化，计算量极大，在大模型上跑不动。

**PPO（近端策略优化）就是TRPO的极简工程化版本**：它不用复杂的二阶优化，只用一个clip裁剪函数，就把策略更新限制在了信赖域里，计算量小、训练稳定。

$$
L^{CLIP}(\theta) = \mathbb{E}_{t} \left[ \min \left( r_t(\theta) \cdot \hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \cdot \hat{A}_t \right) \right]
$$

1. **r_t(θ) = π_θ(a_t|s_t) / π_{θ_old}(a_t|s_t)**：概率比，即「新策略（当前要更新的模型）在状态s_t下选动作a_t的概率」除以「旧策略（采样序列时的模型）选同一个动作的概率」。r_t=1代表策略无变化，r_t>1代表提高了该动作的概率，r_t<1代表降低了概率。
2. **ϵ**：裁剪超参数，默认取0.2，即把r_t限制在0.8~1.2之间。这就是「近端」的含义：每次更新，新策略必须离旧策略很近，从而解决步长失控的问题。
3. **Â_t**：优势函数，$$\hat{A}_t = Q(s_t,a_t) - V(s_t)$$，动作价值函数 - 状态价值函数。直观理解就是「这个动作a_t，比当前状态下的平均动作水平好多少」。
   - Â_t>0：这个动作比平均水平好，要提高它的概率；
   - Â_t<0：这个动作比平均水平差，要降低它的概率。

然而，V(s_t)是无法直接计算的，必须训练一个模型来估计它 —— 也就是 Critic 模型。
- 同样用SFT模型初始化，把lm_head换成输出单个标量的线性层，输入是状态s_t（当前上下文），输出是V(s_t)的估计值；
- 训练Critic的损失是均方误差MSE：让Critic估计的V(s_t)，尽可能接近实际的累计回报G_t；
- 用优势函数替代原始的累计回报G_t，相当于给梯度减去了一个基线，极大降低了梯度的方差，让训练稳定性提升了一个量级。

说到这，我们提到了3个模型，重点解释了其中的2个模型，但是PPO其实是有4个模型：

| 模型名称 | 别名 | 是否固定参数 | 核心作用 |
|----------|------|--------------|----------|
| Policy Model | Actor/策略模型 | 不固定，核心优化目标 | 要优化的LLM，负责生成回答，对应RL的策略π_θ |
| Reference Model | 参考模型 | 固定（SFT后原始模型） | 计算KL散度，约束策略不偏离初始分布，防止模式崩溃 |
| Reward Model | RM/奖励模型 | 固定（阶段2训练完成） | 给生成的完整回答打偏好分，提供核心奖励信号 |
| Value Model | Critic/价值模型 | 不固定，和Policy同步更新 | 估计状态价值V(s_t)，计算优势函数，降低梯度方差，提升训练稳定性 |

也就是说，给定一个 Reference Model，我通过「采样-更新」迭代 Policy Model，与此同时需要 Value Model 估计状态价值，但是完整的奖励公式其实是两部分：
$$R_{total} = R_{RM}(y|x) - \beta \cdot D_{KL}(\pi_{ref}(y|x) || \pi_\theta(y|x))$$
即 Reward Model 估计的奖励分数和一个 KL 散度惩罚项（β是超参）。
1. **防止模式崩溃**：没有KL约束，模型会为了最大化RM分数，学会投机取巧（比如只生成RM喜欢的固定句式，哪怕内容错误，或者重复输出），彻底失去多样性和基础能力，这就是模式崩溃。KL约束强制模型不能偏离初始的SFT模型太远，保住模型的基础能力。
2. **给RM的错误兜底**：RM不是完美的，可能会给错误的回答打高分，KL约束能防止模型为了迎合RM的错误，彻底学坏。

> 至于RM是怎么训练得到的，不在引言的责任范围内！

#### 训练循环
##### 第一步：Rollout（采样阶段）
1. 从prompts库采样一批prompt；
2. 用当前Policy模型，给每个prompt自回归一个完整的回答序列，记录每一步的状态、动作、动作概率；
3. 对每个生成的序列：
   - 用固定RM计算偏好奖励R_RM；
   - 用固定Reference模型计算序列的KL散度，得到KL惩罚项；
   - 计算最终总奖励R_total = R_RM - β·KL；
4. 用当前Value模型，给序列的每一个状态估计V(s_t)；
5. 用R_total和V(s_t)，计算每一步的优势函数Â_t和累积回报G_t。

##### 第二步：Update（更新阶段）
1. 用采样得到的(s_t, a_t, Â_t)，计算PPO clip损失，更新Policy模型参数；
2. 用采样得到的(s_t, G_t)，计算MSE损失，更新Value模型参数；
3. 重复 **Rollout-Update** 循环，直到模型收敛。

#### 适用场景与痛点
- 适用场景：**主观偏好类任务**，比如对话的helpfulness、harmlessness、风格对齐。这类任务没有客观对错，只能靠人类偏好打分，RM能精准捕捉这种主观偏好。
- 核心痛点：
  1. 工程复杂度极高：需要同时加载4个大模型，分布式训练、超参数调试难度极大；
  2. 显存/计算开销爆炸：4个几十B的模型同时加载，哪怕用LoRA，显存开销也远超SFT；
  3. 训练不稳定：极易出现KL爆炸、模式崩溃、灾难性遗忘（模型忘记预训练的基础能力）；
  4. 推理任务效果差：对于有客观标准答案的数学/代码任务，RM很难精准判断答案对错，也无法给长推理链的中间步骤打分，PPO无法有效优化推理能力。

### DPO
PPO的所有痛点，根源都是把RLHF拆成了「训练RM」和「RL优化策略」两个独立阶段，流程极其绕。

DPO的核心贡献，是从理论上证明了 **RLHF的最优策略，存在一个闭式解析解，可以直接把RL优化目标，转化成一个简单的监督式损失，完全不需要训练RM、Critic模型，也不需要RL的采样-更新循环**。

其实原本就是个优化问题，RLHF的终极优化目标，是找到一个最优策略π*，在最大化RM奖励的同时，和参考模型π_ref的KL散度不超过阈值：
$$\pi^* = \arg\max_\pi \mathbb{E}_{(x,y) \sim \pi} [ R_{RM}(x,y) ] - \beta D_{KL}(\pi(y|x) || \pi_{ref}(y|x))$$

DPO通过最大熵RL理论，证明了这个优化问题的最优策略，和RM有严格的一一对应关系：
$$\pi^*(y|x) = \frac{1}{Z(x)} \pi_{ref}(y|x) \exp\left( \frac{1}{\beta} R_{RM}(x,y) \right)$$
其中Z(x)是归一化的配分函数。

把这个式子变形，就能把RM的奖励，用最优策略和参考模型的概率比表示出来：
$$R_{RM}(x,y) = \beta \log \left( \frac{\pi^*(y|x)}{\pi_{ref}(y|x)} \right) + \beta \log Z(x)$$

原来的RM训练，是用成对偏好数据(x, y_w, y_l)（y_w是人类偏好的回答，y_l是不偏好的），目标是让R(x,y_w) > R(x,y_l)。

把上面的R表达式代入排序损失，神奇的事情发生了：**同一个prompt的配分函数Z(x)会完全抵消，β是常数可以提取，整个损失函数里，彻底没有了RM的影子，只剩下最优策略π*和参考模型π_ref**。

这就是DPO的最终损失函数：
$$L_{DPO}(\pi_\theta) = - \mathbb{E}_{(x,y_w,y_l) \sim D} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)} \right) \right]$$

直观的说，对于同一个prompt的两个回答：
- 我们要让**当前模型给好回答y_w的概率，相对于参考模型的提升幅度，远远大于给坏回答y_l的概率的提升幅度**。
- 换句话说：好的回答，模型要比参考模型更愿意生成；坏的回答，模型要比参考模型更不愿意生成。
- 本质就是一个**二分类任务**：模型要学会区分「好回答」和「坏回答」，给好回答更高的生成概率，这就是原文说的「分类式损失」。

| 维度 | PPO | DPO |
|------|-----|-----|
| 所需模型 | 4个（Policy、Reference、RM、Value） | 2个（Policy、Reference，甚至可以共享权重，只开两次前向） |
| 训练流程 | 复杂的「采样-更新」RL循环，需要自回归生成序列 | 纯监督式训练，和SFT几乎完全一致，只需要成对偏好数据，无需采样生成 |
| 工程门槛 | 极高，需要复杂的分布式框架，超参数调试难度大 | 极低，SFT训练代码改个损失函数就能实现 |
| 显存开销 | 极大，4个大模型同时加载 | 极小，仅需加载1个模型，显存开销和SFT相当 |
| 训练稳定性 | 极差，极易KL爆炸、模式崩溃 | 极好，和监督学习一样稳定，几乎不会训崩 |
| 偏好对齐效果 | 好，但不稳定 | 和PPO相当，甚至更优，一致性更强 |

当然当然，DPO「不天然擅长客观可验证的搜索型推理」，这里补全4个核心原因，也是它无法替代GRPO的关键：
1. **无信用分配能力，无法优化延迟奖励**：DPO是对完整序列做损失，只能学到「整个y_w比y_l好」，但无法知道y_w里哪一步推理是关键、哪一步是对的，无法给中间步骤分配信用。而数学推理的核心，就是中间步骤的优化，DPO完全做不到。
2. **无在线探索能力，只能模仿离线数据**：DPO的训练数据是提前准备好的成对偏好数据，模型只能学数据里有的内容，无法自主探索训练集之外的解题思路。而难题的核心，就是要探索训练集没有的推理路径，DPO天生不具备这个能力。
3. **对相对偏好强，对绝对正确弱**：DPO只能学到「y_w比y_l好」，但无法保证y_w是绝对正确的。比如y_w是错的但通顺的回答，y_l是更错的回答，DPO会让模型更愿意生成y_w，但它依然是错的。而数学任务需要的是「绝对正确」，不是「相对更好」。
4. **抑制长程推理**：DPO的KL约束，会让模型尽量生成和参考模型分布接近的序列，长推理链的KL散度天然更大，模型会本能地不愿意生成更长的CoT，而长CoT是解决复杂难题的核心。

### GRPO
推理任务最终要「回到RL」，SFT/DPO天生无法满足：
. **奖励可100%自动化、客观化验证**：不需要人类打分，不需要训练RM，我们可以用程序直接判断对错——数学题可以对比标准答案/用符号计算验证，代码可以跑单元测试，逻辑题可以校验规则。这个奖励是完全客观、无偏差的，比RM精准无数倍。
2. **必须解决长程信用分配**：推理任务的对错，只有生成完整的CoT和答案后才知道，中间的每一步推理，单独看没有对错，只有放到完整解题路径里才有意义。只有RL能把最终的对错奖励，精准分配到每一步的推理动作上，让模型知道「哪一步该走，哪一步该停」。
3. **必须有在线探索能力**：复杂难题的解题思路，不可能全部出现在训练集里。SFT/DPO只能模仿训练集的示范，而RL可以让模型自主采样不同的推理路径，通过「对了给奖励，错了无奖励」的反馈，探索出训练集之外的更优解法，这也是DeepSeek-R1能突破闭源模型的核心。

GRPO（Group Relative Policy Optimization，组相对策略优化），最早在DeepSeekMath论文中提出，后续在DeepSeek-R1中大规模落地，是当前推理RL的工业级标准方案。它的核心突破，是 **用「组内相对归一化的优势函数」，完全替代了PPO里的Critic模型**，同时保留了PPO的训练稳定性，去掉了所有冗余组件，适配推理任务。

#### 比PPO好在哪
##### 改动1：去掉Critic模型，用组内相对奖励计算优势函数
对同一个prompt x，一次性采样G个不同的完整回答序列（组成一个Group，G一般取4~16），每个序列用程序化奖励函数算出总奖励R_i；

组内相对优势函数计算：
$$\hat{A}_i = \frac{R_i - \mu_R}{\sigma_R + \epsilon}$$
其中μ_R是组内所有奖励的平均值，σ_R是组内奖励的标准差，ϵ是防止除零的小常数。

这个优势函数，就是「这个序列的奖励，在同一个prompt的所有采样结果里，比平均水平好多少」。比如同一个prompt采样4个序列，2个答对（R=1），2个答错（R=0），那么答对的序列Â_i为正，答错的为负，模型会自然提高正确路径的概率，降低错误路径的概率。

同一个prompt的所有采样序列，初始状态完全一致，它们的状态价值V(s_0)是相同的，用组内平均奖励就能精准估计这个基线，完全不需要单独的Critic模型。

##### 改动2：逐token的KL正则替代全局KL惩罚
GRPO没有用PPO的全局KL惩罚，而是把KL约束融入到每一步的损失函数中，逐token计算。这样既能防止模型偏离参考模型太远，又不会抑制长CoT的生成——只要每一步的token概率在合理范围内，哪怕序列很长，也不会有过大的KL惩罚，鼓励模型做更长的推理。

##### 改动3：保留PPO的clip机制，保证训练稳定
GRPO保留了PPO的核心clip裁剪机制，限制策略的更新幅度，避免步长失控，同时用组内相对优势函数，让梯度更精准，训练比PPO更稳定。

#### 目标函数
$$
L_{GRPO}(\theta) = \mathbb{E}_{x \sim D} \mathbb{E}_{(y_1,...,y_G) \sim \pi_\theta(\cdot|x)} \left[ \frac{1}{G} \sum_{i=1}^G \sum_{t=1}^{T_i} \min \left( r_{i,t}(\theta) \cdot \hat{A}_i, \text{clip}(r_{i,t}(\theta), 1-\epsilon, 1+\epsilon) \cdot \hat{A}_i \right) - \beta \cdot D_{KL}(\pi_\theta(\cdot|s_{i,t}) || \pi_{ref}(\cdot|s_{i,t})) \right]
$$

#### 训练循环
##### 第一步：Group Rollout（组采样阶段）
1. 从prompts库采样一个batch的数学题prompt；
2. 对每个prompt，用当前Policy模型，并行采样G个不同的完整回答序列（包含CoT推理过程和最终答案）
3. 对每个生成的序列，用 **程序化奖励函数** 计算总奖励R_i：
    - 核心硬奖励：正确性奖励，答案正确给1分，错误给0分；
    - 辅助软奖励：CoT长度奖励（鼓励合理的长推理）、格式奖励（鼓励先推理后作答的规范格式）、重复惩罚（惩罚重复输出）、步骤有效性奖励（惩罚无效的凑数步骤）；
4. 对每个prompt的组内G个奖励，计算平均值和标准差，计算出前面提到的每个序列的相对优势Â_i；
5. 对每个序列的每个token，用固定的Reference模型计算逐token的KL散度用于正则项。

##### 第二步：Update（更新阶段）
1. 用采样得到的所有序列的token、概率比、优势Â_i，计算GRPO的目标函数；
2. 用梯度下降更新Policy模型的参数；
3. 重复「采样-更新」循环，直到模型的解题准确率收敛。

当然现在的开源模型已经不再把“思考”当成单一固定模板。很多新开源/开放权重模型已经不再默认“所有请求都强制长 CoT”，而是开始做 **thinking / non-thinking 双模式**。当年 Deepseek 2024 年的时候前端没优化，流式输出一大堆 CoT 能把浏览器卡死。
换句话说应该叫 **hybrid reasoning**，让同一个模型能在简单任务里快答，在复杂任务里展开推理。

| 阶段                  | 学什么         | 数据长什么样              | 优点                  | 局限           |
| ------------------- | ----------- | ------------------- | ------------------- | ------------ |
| Pre-train           | 世界分布、语言规律   | 海量无标注文本             | 覆盖广、能力底座强           | 不懂指令，领域密度稀   |
| Continue Pre-train  | 领域知识密度      | 高质量专业语料             | 真正把领域知识写进参数         | 仍然只是“会续写”    |
| SFT                 | 行为协议、回答格式   | 高质量指令-回答对           | 快速把模型拉成助手           | 更像模仿，不擅长搜索   |
| PPO / RLHF          | 人类主观偏好      | 排序数据 + RM           | 能优化 help/safe/style | 工程复杂、显存重     |
| DPO                 | 直接偏好对齐      | chosen / rejected 对 | 稳、简单、便宜             | 不擅长在线探索      |
| Reasoning RL / GRPO | 可验证任务上的搜索能力 | 可自动打分任务             | 对数学/代码很有效           | 依赖可验证 reward |

## 1.2 训练数据
### 1.2.1 结构化推理
RL 训练需要批量生成、处理成千上万的样本，全程无人工干预。如果模型自由输出长篇大论，解析程序（Parser）无法稳定、100% 准确地从文本中定位「最终答案到底是哪一段」。

```
response
  ├─ <think> ... </think>   # 推理过程
  └─ <answer> ... </answer> # 最终答案
```

SFT+RL 反复让模型看到题目、先进入 `<think>`、再输出 `<answer>`。久而久之，模型会把这个格式内化成条件反射，只要看到，就自动切换到「分步解题模式」。

标签把「推理过程 token」和「最终答案 token」严格分离，我们可以精准控制奖励的作用范围：不会让推理过程的有效试错，被最终答案的错误过度惩罚；也不会让最终答案的正确，被推理过程的无效内容稀释，让信用分配更合理，训练收敛更快。

具体到大一点的提示词方面，SFT 和 RL 阶段必须使用完全一致的 Prompt 模板，保证模型在两个阶段看到的输入格式、指令要求完全相同。
如果 SFT 用一套提示词，RL 换另一套，模型的输出行为会发生偏移，原本学会的格式规范、推理流程会失效，直接导致 RL 阶段生成大量无效样本，训练无法收敛。

并且，固定模板通过明确的指令，解决了模型闲聊、跳步、输出不可解析的问题：
- 角色约束：明确对话双方的身份和核心任务，解决「我是谁，我要做什么」的角色混乱问题；
- 过程约束：强制要求「先思考，后回答」的流程，解决模型跳步、不展开 CoT 的问题；
- 可解析约束：明确要求最终答案的格式规范，解决模型输出冗余、无法解析的问题。
- 无需猜测「我该以什么格式输出」，只需要专注于推理内容本身，大幅降低了学习难度，让模型更快收敛到符合要求的输出行为上。

In [ ]:
PROMPT_TEMPLATE = """A conversation between User and Assistant. The User asks a question, and the Assistant solves it.
The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer.
The reasoning process is enclosed within <think> </think> and the answer is enclosed within <answer> </answer> tags, respectively.
The final result only requires a direct output of the answer without any explanations.

User: {question}
Assistant: <think>
"""

| 模板段落 | 对应约束 | 核心设计目的 |
|----------|----------|--------------|
| 第一段：`A conversation between User and Assistant...` | 角色约束 | 明确对话双方身份，锁定「用户提问，助手解题」的核心任务，避免模型角色错位、输出无关闲聊内容 |
| 第二段：`The Assistant first thinks about the reasoning process...` | 过程约束 | 明确「先思考，后回答」的核心流程，给模型注入推理行为的先验 |
| 第三段：`The reasoning process is enclosed within...` | 格式约束 | 明确标签的使用规则，和结构化标签体系对齐，保证输出可被解析 |
| 第四段：`The final result only requires a direct output...` | 可解析约束 | 强制要求最终答案无冗余解释，避免模型在内输出废话，导致解析失败、等价性判定错误 |
| 结尾：`Assistant: <think>` | 强制行为约束 | 预填起始标签，把模型的输出起点直接拉到推理区间，从根源上避免模型跳过思考、直接输出答案或闲聊 |


> **强制所有样本都这么做，不一定是长期最优设计**。因为简单题也被迫展开长推理，推理成本和延迟都会上升。

### 1.2.2 规则奖励函数设计
规则奖励函数是 Reasoning RL 的奖励信号来源，它替代了 RLHF 中的奖励模型（RM），通过 Python 脚本实现程序化、客观、无偏差的打分，是整个 RL 训练闭环的核心。
#### 优势
1. **100% 客观公正，奖励信号更干净**
数学题的对错有唯一的客观标准，不需要人类主观偏好。RM 是基于人类标注训练的，天生带有标注偏差、打分错误，甚至会给错误但通顺的回答打高分；而规则奖励完全基于客观规则判定，不会出错，给模型的奖励信号更纯粹，训练出来的推理能力更扎实。
2. **工程成本极低，训练效率更高**
RM 需要收集大量人类偏好排序数据、单独训练和部署一个大模型，显存、计算、人力成本极高；而规则奖励仅需一个 Python 脚本，就能实现批量、实时的打分，无需额外的模型部署，成本降低，适配 RL 阶段海量样本的处理需求。
1. **彻底避免奖励黑客（Reward Hacking）**
RM 很容易被模型找到漏洞 —— 模型会学会迎合 RM 的打分偏好，而非输出正确答案（比如生成 RM 喜欢的句式，哪怕答案错误）；而规则奖励的判定标准是固定、刚性的，模型无法投机取巧，只能通过输出正确的答案拿到奖励，从根源上避免奖励黑客。

#### 两级闸门
```
response
   ↓
[格式检查]
有没有 <think> / <answer> / 闭合是否正确
   ↓
[答案提取]
抽出 model_answer
   ↓
[等价性判定]
与 ground_truth 是否数学等价
   ↓
reward ∈ {0, 1} 或更细粒度分数
```

1. **第一闸：格式奖励（Format Reward）**
- 完整 response 必须同时包含成对的和标签，无遗漏、无嵌套、无颠倒；
- 标签顺序必须严格符合顺序，禁止推理内容和答案出现在对方的标签内；
- 标签必须正确闭合，禁止出现单标签、标签拼写错误。

格式完全合规，格式奖励记 1.0；任何一项不满足，格式奖励记 0.0，且总奖励直接置 0，无需进入后续答案判定。

In [ ]:
import re
import math
from typing import Optional
from sympy import sympify, simplify

def extract_between(text: str, start_tag: str, end_tag: str) -> Optional[str]:
    # re.escape(start_tag)：转义标签中的特殊字符（如 "<" ">"），避免被正则识别为语法
    # r"(.*?)"：非贪婪匹配任意字符（.*? 表示匹配尽可能短的内容，防止跨标签匹配）
    # 正则表达式里，用圆括号 () 包起来的部分，叫「捕获组」（Capture Group），它的作用是
    # 先匹配整个正则，再把括号里匹配到的内容「单独存起来」，供你后续提取
    pattern = re.secape(start_tag) + r"(.*?)" + re.escape(end_tag)

    # re.DOTALL：让 "." 也能匹配换行符（因为推理过程和答案通常是多行的）
    # 如果匹配成功：返回一个 Match 对象
    m = re.search(pattern, text, re.DOTALL)

    # Match 对象有一个 group() 方法，用来提取捕获组的内容
    # m.group() 或 m.group(0)	提取整个正则匹配到的完整内容
    # 这里group(1)表示提取第 1 个捕获组的内容
    # 我们这里没有第 2 个括号，不能m.group(2)
    result = m.group(1).strip() if m else None
    # m.group()得到一个str，str.strip()会去除字符串「首尾」的所有空白字符，就是空格换行那些
    # 然而字符串中间的空白不会被去掉
    return result


2. **第二闸：答案奖励（Answer Reward）**
- **第一步：答案归一化**
  - 先把不同写法的答案（比如1/2、0.5、2/4、\frac{1}{2}），清洗成统一的标准化格式，消除文本、LaTeX、格式带来的噪声，为后续等价判定打下基础。
  - 优先从\boxed{}中提取内容（数学竞赛标准答案的通用格式），再从标签中提取核心文本；
  - 删除 LaTeX 冗余标签（\text{}、\mathrm{}等）、多余空格、换行、单位（miles、kg 等）、标点符号；
  - 统一分数、小数、括号的写法，统一大小写，消除格式差异。


In [1]:
def normalize_math_answer(ans:str) -> str:
    ans = ans.strip()

    # 去除LaTeX的boxed标签（数学竞赛/标准答案的通用格式，如 \boxed{72}）
    # 先替换左标签，再替换右花括号
    ans = ans.replace("\\boxed{", "").replace("}", "") if "\\boxed{" in ans else ans

    # 去除LaTeX行内数学公式标记
    ans = ans.replace("$", "")  # ctrl+H 把$替换成空字符串

    # 去除LaTeX的文本标签\text{}
    ans = re.sub(r"\\text\{.*?\}", "", ans)

    # \s：匹配任何空白字符（包括空格、制表符 \t、换行符 \n、回车符 \r、换页符 \f 等）
    # +：表示匹配一个或多个连续的空白字符。
    ans = re.sub(r"\s+", "", ans)

    return ans

- **第二步：符号等价性判定**
  - 解决「形式不同但代数等价」的问题（比如x+y和y+x、(a+b)^2和a^2+2ab+b^2），这是字符串对比无法实现的。
  - 把归一化后的两个答案，转换为 SymPy 的符号表达式；
  - 计算两个表达式的差值，若差值化简后恒等于 0，则判定两个表达式代数等价；
  - 兼容整数、分数、代数表达式、方程、不等式等多种数学形式。


In [ ]:

def symbolic_equal(a:str, b:str) -> bool:
    """
    基于SymPy符号计算库，判断两个数学表达式是否代数等价
    核心作用：解决"形式不同但代数等价"的问题（如 "x+y" 和 "y+x"，"(a+b)^2" 和 "a^2+2ab+b^2"）
    
    参数:
        a: 标准化后的答案1
        b: 标准化后的答案2
    
    返回:
        代数等价返回True，否则返回False（解析失败也返回False）
    """
    try:
        # sympify(): 把字符串转化成SymPy的符号表达式
        # simplify(): 代数化简
        return simplify(sympify(a) - sympify(b)) == 0
    except Exception:
        return False
    # 调库达人

- **第三步：数值近似判定**
  - 解决浮点数精度误差问题（比如1/3和0.333333），符号计算可能无法识别这类近似等价的情况，必须通过数值计算补充判定。
  - 基于math.isclose实现，设置合理的误差阈值：
    - 把两个答案转换为浮点数，计算相对误差；
    - 通用阈值设置：相对误差rel_tol=1e-4（万分之一），绝对误差abs_tol=1e-9，在误差范围内则判定为等价。
> SymPy 很强，但也可能在一些复杂表达式上卡住。
> 训练中一旦某次等价判定死循环，整个 rollout 就被拖死。所以很多实现里会给符号化简加超时保护。

In [ ]:
def numeric_equal(a:str, b:str, rel_tol: float=1e-4) -> bool:
    # rel_tol: 相对误差容忍度，默认万分之一的误差范围内都算对
    try:
        return math.isclose(float(a), float(b), rel_tol=rel_tol)
    except Exception:
        return False

把上面三个加起来：

In [ ]:
def grade_answer(pred: str, gold: str) -> bool:
    pred = normalize_math_answer(pred)
    gold = normalize_math_answer(gold)

    if pred == gold:
        return True
    
    if symbolic_equal(pred, gold):
        return True
    
    if numeric_equal(pred, gold):
        return True
    
    # 三级判定都不通过，判断为答案错误
    return False

In [ ]:
def reward_fn(response: str, ground_truth: str):
    think = extract_between(response, "<think>", "<\think>")
    answer = extract_between(response, "<answer>", "<\answer>")

    # 一级闸门：只有同时提取到两者内容才算格式合格
    format_ok = (think is not None) and (answer is not None)
    if not format_ok:
        return {
            "format_reward": 0.0,
            "answer_reward": 0.0,
            "reward": 0.0,
        }
    
    # 二级闸门
    is_correct = grade_answer(answer, ground_truth)
    if not is_correct:
        return {
            "format_reward": 1.0,
            "answer_reward": 0.0,
            "reward": 0.0,
        }
    
    return {
        "format_reward": 1.0,
        "answer_reward": 1.0,
        "reward": 1.0,
    }

## 1.3 GSM8K 数据预处理
它们没有开源MATH-12K数据集，所以只能找一些替代的数学竞赛题。这里没用那些已经预处理好了的jsonl数据，也许这之外的数据集我会用，但是至少这里我想亲自体验一下这个过程。
原版属于Parquet格式，分为两个版本：
- main/: 标准版本
- socratic/: 带苏格拉底式引导问题的版本（适合教学场景）

In [4]:
import pandas as pd

# 读取 main 版本（或 socratic 版本）
df = pd.read_parquet(r'D:\MyLab\cs336\datasets\openai_gsm8k\main')

# 查看数据
print(df.columns)  # ['question', 'answer']
print(len(df))     # train: 7473, test: 1319

# 访问样本
question = df.iloc[0]['question']
answer = df.iloc[0]['answer']

# 打印第一个样本
print("=" * 50)
print("第1个样本:")
print("=" * 50)
print("\n【Question】")
print(df.iloc[0]['question'])
print("\n【Answer】")
print(df.iloc[0]['answer'])

Index(['question', 'answer'], dtype='object')
8792
第1个样本:

【Question】
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

【Answer】
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


可以看到，数据结构为：
- question: 数学应用题
- answer: 分步解答，格式为推理过程 + #### 最终答案


In [7]:
import pandas as pd
import re
import json
from pathlib import Path


def extract_final_answer(answer_text: str) -> str:
    """
    从GSM8K答案中提取最终答案（####后面的数字）
    例如："...\n#### 18" -> "18"
    """
    # 匹配 #### 后面的数字（支持负数、小数、分数）
    match = re.search(r'####\s*(-?\d+\.?\d*(?:/\d+)?)', answer_text.strip())
    if match:
        return match.group(1).strip()
    return answer_text.strip().split('\n')[-1].strip()


def convert_to_structured_format(question: str, answer: str) -> dict:
    """
    将GSM8K原始格式转换为结构化推理格式
    输入: question + answer（包含推理过程和####最终答案）
    输出: 带<think>和<answer>标签的完整回复
    """
    # 提取推理过程（####之前的部分）
    think_content = answer.split('####')[0].strip()

    # 清理推理过程中的特殊标记 <<...=...>>
    think_content = re.sub(r'<<[^>]+=([^>]+)>>', r'\1', think_content)

    # 提取最终答案
    final_answer = extract_final_answer(answer)

    # 构建结构化输出
    structured_response = f"""<think>
{think_content}
</think>
<answer>
{final_answer}
</answer>"""

    return {
        "question": question,
        "ground_truth": final_answer,
        "structured_response": structured_response,
        "think": think_content,
        "answer": final_answer
    }


def preprocess_gsm8k(parquet_path: str, output_path: str, split: str = "train"):
    """
    预处理GSM8K数据集，转换为适合RL训练的JSONL格式

    输出格式（每行一个JSON）：
    {
        "prompt": "完整的prompt（含PROMPT_TEMPLATE）",
        "response": "带<think>和<answer>标签的完整回复",
        "ground_truth": "标准答案（用于奖励函数计算）"
    }
    """
    # 读取parquet文件
    df = pd.read_parquet(parquet_path)
    print(f"[{split}] 原始数据量: {len(df)}")

    PROMPT_TEMPLATE = """A conversation between User and Assistant. The User asks a question, and the Assistant solves it.
The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer.
The reasoning process is enclosed within <think> </think> and the answer is enclosed within <answer> </answer> tags, respectively.
The final result only requires a direct output of the answer without any explanations.

User: {question}
Assistant: <think>
"""

    processed_data = []

    for idx, row in df.iterrows():
        question = row['question']
        answer = row['answer']

        # 转换为结构化格式
        structured = convert_to_structured_format(question, answer)

        # 构建完整的prompt（注意已经预填了<think>标签开头）
        prompt = PROMPT_TEMPLATE.format(question=question)

        # 构建训练样本
        sample = {
            "prompt": prompt,
            "response": structured["structured_response"],
            "ground_truth": structured["ground_truth"],
            "raw_question": question,
            "raw_answer": answer
        }
        processed_data.append(sample)

    # 保存为JSONL格式
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    with open(output_file, 'w', encoding='utf-8') as f:
        for sample in processed_data:
            f.write(json.dumps(sample, ensure_ascii=False) + '\n')

    print(f"[{split}] 预处理完成，保存到: {output_file}")
    print(f"[{split}] 总样本数: {len(processed_data)}")

    return processed_data


In [8]:
# 配置路径
base_dir = Path(r"D:\MyLab\cs336\datasets\openai_gsm8k")
output_dir = Path(r"D:\MyLab\cs336\datasets\openai_gsm8k\processed")

# 预处理训练集
train_data = preprocess_gsm8k(
    parquet_path=str(base_dir / "main" / "train-00000-of-00001.parquet"),
    output_path=str(output_dir / "train.jsonl"),
    split="train"
)

# 预处理测试集
test_data = preprocess_gsm8k(
    parquet_path=str(base_dir / "main" / "test-00000-of-00001.parquet"),
    output_path=str(output_dir / "test.jsonl"),
    split="test"
)

print("\n全部预处理完成！")
print(f"训练集: {len(train_data)} 条")
print(f"测试集: {len(test_data)} 条")

[train] 原始数据量: 7473
[train] 预处理完成，保存到: D:\MyLab\cs336\datasets\openai_gsm8k\processed\train.jsonl
[train] 总样本数: 7473
[test] 原始数据量: 1319
[test] 预处理完成，保存到: D:\MyLab\cs336\datasets\openai_gsm8k\processed\test.jsonl
[test] 总样本数: 1319

全部预处理完成！
训练集: 7473 条
测试集: 1319 条
